# 🚀 Тестирование Agent Framework

Этот ноутбук пошагово проверяет все компоненты фреймворка агентов:

1. **Тест базовых функций** - проверка инструментов
2. **Test Agent** - простейший агент (1 состояние)
3. **My Agent** - переходы между состояниями (2 состояния)
4. **Router Agent** - условный роутинг (ветвление)
5. **Multi-Agent System** - один агент вызывает другого
6. **Audit Agent** - полный тест (6 состояний)

Бэкенд (GigaChat или LM Studio) настраивается в `config.yaml`.

In [1]:
# Установка зависимостей (запустите один раз)
# %pip install -r requirements.txt

## 📦 Общая настройка

Импорты и конфигурация для всех тестов.

In [2]:
# Импорты
import yaml
from pprint import pprint

# Импорты агентов
from agents.test_agent import TestAgent
from agents.my_agent import MyAgent
from agents.router_agent import RouterAgent
from agents.supervisor_agent import SupervisorAgent
from agents.audit_agent import AuditAgent

# Импорты инструментов и клиентов
from tools.tools import (
    tools, 
    multiagent_tools,
    get_tools_dict, 
    reset_memory,
    register_agent,
    calculator
)
from connections.clients import get_llm_client

# Для логирования
from agent_engine.debug import enable_logging, disable_logging

print("✓ Импорты выполнены успешно")

✓ Импорты выполнены успешно


In [3]:
# Загрузка конфигурации
with open("config.yaml", "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

backend = config['active_backend']
recursion_limit = config['agent']['recursion_limit']

print(f"Активный бэкенд: {backend}")
print(f"Лимит рекурсии графа: {recursion_limit}")

Активный бэкенд: lmstudio
Лимит рекурсии графа: 30


In [4]:
# Создание LLM клиента
llm = get_llm_client(backend, config)
print(f"✓ LLM клиент создан: {type(llm).__name__}")

✓ LLM клиент создан: ChatOpenAI


In [5]:
# Включение логирования (опционально)
enable_logging()
print("✓ Логирование включено")

# Для отключения раскомментируйте:
# disable_logging()
# print("✓ Логирование отключено")

✓ Логирование включено


---

## 1️⃣ Тест базовых функций

Проверяем что инструменты работают корректно.

In [7]:
print("=" * 60)
print("ТЕСТ 1: Базовые функции")
print("=" * 60)

# Тест калькулятора
print("\n1. Тест калькулятора:")
result = calculator.invoke("2 ** 10")
print(f"   2^10 = {result}")
assert result == "1024", "Калькулятор работает неправильно!"
print("   ✓ Калькулятор работает")

# Тест памяти
print("\n2. Тест памяти:")
from tools.tools import memory
reset_memory()
memory.invoke({"action": "save", "key": "test_key", "value": "test_value"})
result = memory.invoke({"action": "get", "key": "test_key"})
assert "test_value" in result, "Память работает неправильно!"
print("   ✓ Память работает")

print("\n✅ Все базовые функции работают корректно!")

ТЕСТ 1: Базовые функции

1. Тест калькулятора:
[TOOL] calculator params={"expression": "2 ** 10"}
[TOOL] memory_append params={"text": "[calculator] 2 ** 10 = 1024"}
   2^10 = 1024
   ✓ Калькулятор работает

2. Тест памяти:
[TOOL] memory params={"action": "save", "key": "test_key", "value": "test_value"}
[TOOL] memory params={"action": "get", "key": "test_key", "value": ""}
   ✓ Память работает

✅ Все базовые функции работают корректно!


---

## 2️⃣ Test Agent - Простейший агент

Граф: `[work] → END`

Проверяем:
- Создание агента через AgentConfig
- Работу с инструментами
- Переход в END по ключевому слову

In [8]:
print("=" * 60)
print("ТЕСТ 2: Test Agent (1 состояние)")
print("=" * 60)

# Подготовка
reset_memory()
tools_dict = get_tools_dict(tools)

# Создание агента
test_agent = TestAgent(llm, tools_dict)
print(f"\n✓ Агент создан: {test_agent}")

# Визуализация графа
print("\nСтруктура графа:")
print(test_agent.visualize())

ТЕСТ 2: Test Agent (1 состояние)

✓ Агент создан: TestAgent(id=TestAgent_2150759280960, states=1)

Структура графа:
📊 Граф агента:

Состояния:
  • work (entry)
    Инструменты: calculator, memory, think
    Описание: Единственное рабочее состояние

Переходы:
  • work → END (условный)
    Завершение работы по ключевому слову


In [9]:
# Запуск агента
print("\n" + "=" * 60)
print("Запуск: Посчитай 15 * 23 и сохрани результат в память")
print("=" * 60 + "\n")

result = test_agent.invoke([
    "Посчитай 15 * 23 и сохрани результат в память с ключом 'result'"
])

# Вывод результата
print("\n" + "=" * 60)
print("Результат:")
print("=" * 60)
print(result['messages'][-1].content)

print("\n✅ Test Agent работает корректно!")


Запуск: Посчитай 15 * 23 и сохрани результат в память

[STATE] START -> work
[TOOL] calculator params={"expression": "15 * 23"}
[TOOL] memory_append params={"text": "[calculator] 15 * 23 = 345"}
[TOOL] memory params={"action": "save", "key": "result", "value": "345"}

Результат:
ГОТОВО

✅ Test Agent работает корректно!


---

## 3️⃣ My Agent - Переходы между состояниями

Граф: `[work] ⟳ → [summarize] → END`

Проверяем:
- Циклическое состояние (work возвращается в себя)
- Условный переход по ключевому слову
- Работу с памятью между состояниями

In [10]:
print("=" * 60)
print("ТЕСТ 3: My Agent (2 состояния + переход)")
print("=" * 60)

# Подготовка
reset_memory()
tools_dict = get_tools_dict(tools)

# Создание агента
my_agent = MyAgent(llm, tools_dict)
print(f"\n✓ Агент создан: {my_agent}")

# Визуализация графа
print("\nСтруктура графа:")
print(my_agent.visualize())

ТЕСТ 3: My Agent (2 состояния + переход)

✓ Агент создан: MyAgent(id=MyAgent_2150759972496, states=2)

Структура графа:
📊 Граф агента:

Состояния:
  • work (entry)
    Инструменты: calculator, ask_human, think, memory
    Описание: Основное рабочее состояние с полным набором инструментов
  • summarize
    Инструменты: summarize, memory
    Описание: Финальное состояние для рефлексии и подведения итогов

Переходы:
  • work → summarize (условный)
    Переход из рабочего состояния в состояние подведения итогов. Срабатывает, когда агент говорит 'ЗАДАЧА_РЕШЕНА'. Если условие не выполнено, агент остается в Work и продолжает работу.
  • summarize → END (условный)
    Безусловное завершение работы после подведения итогов. Состояние Summarize всегда переходит в END.


In [11]:
# Запуск агента
print("\n" + "=" * 60)
print("Запуск: Посчитай 2 в степени 10, затем сделай саммари")
print("=" * 60 + "\n")

result = my_agent.invoke([
    # "Посчитай 2 в степени 10"
    "давай подсчитаем площадь круга?"
])

# Вывод результата
print("\n" + "=" * 60)
print("Результат:")
print("=" * 60)
print(result['messages'][-1].content)

print("\n✅ My Agent работает корректно!")


Запуск: Посчитай 2 в степени 10, затем сделай саммари

[STATE] START -> work
[TOOL] ask_human params={"question": "Какой радиус круга?"}

🤔 Вопрос агента: Какой радиус круга?
[TOOL] calculator params={"expression": "3.141592653589793 * 5 ** 2"}
[TOOL] memory_append params={"text": "[calculator] 3.141592653589793 * 5 ** 2 = 78.53981633974483"}
[STATE] work -> summarize
[TOOL] summarize params={"focus": "results"}

Результат:
**Краткое и понятное саммари**

В ходе диалога пользователь запросил вычисление площади круга с радиусом 5 см. Я предоставил результат: площадь ≈ 78.54 кв. см.

---

**Сохранено в память** (ключ `circle_area_summary`):

```
Пользователь попросил рассчитать площадь круга радиуса 5 см.
Ответ: 78,54 кв. см (формула πr²).
```

✅ My Agent работает корректно!


---

## 4️⃣ Router Agent - Условный роутинг

Граф: `[classify] → [math | text | error] → END`

Проверяем:
- Классификацию запроса
- Роутер с несколькими выходами
- Разные пути обработки

In [12]:
print("=" * 60)
print("ТЕСТ 4: Router Agent (условный роутинг)")
print("=" * 60)

# Подготовка
reset_memory()
tools_dict = get_tools_dict(tools)

# Создание агента
router_agent = RouterAgent(llm, tools_dict)
print(f"\n✓ Агент создан: {router_agent}")

# Визуализация графа
print("\nСтруктура графа:")
print(router_agent.visualize())

ТЕСТ 4: Router Agent (условный роутинг)

✓ Агент создан: RouterAgent(id=RouterAgent_2150759964672, states=4)

Структура графа:
📊 Граф агента:

Состояния:
  • classify (entry)
    Инструменты: think, memory
    Описание: Классификация типа запроса
  • math
    Инструменты: calculator, memory, think
    Описание: Обработка математических запросов
  • text
    Инструменты: memory, think
    Описание: Обработка текстовых запросов
  • error
    Инструменты: think
    Описание: Обработка нераспознанных запросов

Переходы:
  • classify → [math→math, text→text, error→error]
    Роутинг по типу запроса после классификации
  • math → END (условный)
    Завершение после обработки математического запроса
  • text → END (условный)
    Завершение после обработки текстового запроса
  • error → END (условный)
    Завершение после обработки ошибки


In [13]:
# Тест 1: Математический запрос
print("\n" + "=" * 60)
print("Тест 4.1: Математический запрос")
print("=" * 60 + "\n")

reset_memory()
result = router_agent.invoke(["Посчитай 5 + 5"])

print("\nРезультат:")
print(result['messages'][-1].content)
print("\n✓ Математический путь работает")


Тест 4.1: Математический запрос

[STATE] START -> classify
[TOOL] think params={"thought": "User asks calculation 5+5, so math."}

💭 Размышление агента: User asks calculation 5+5, so math.
[TOOL] memory params={"action": "save", "key": "request_type", "value": "math"}
[STATE] classify -> math
[TOOL] calculator params={"expression": "5 + 5"}
[TOOL] memory_append params={"text": "[calculator] 5 + 5 = 10"}
[TOOL] memory params={"action": "save", "key": "last_result", "value": "10"}

Результат:
Результат вычисления: **10**  
ОБРАБОТКА_ЗАВЕРШЕНА

✓ Математический путь работает


In [14]:
# Тест 2: Текстовый запрос
print("\n" + "=" * 60)
print("Тест 4.2: Текстовый запрос")
print("=" * 60 + "\n")

reset_memory()
result = router_agent.invoke(["Привет! Как дела?"])

print("\nРезультат:")
print(result['messages'][-1].content)
print("\n✓ Текстовый путь работает")

print("\n✅ Router Agent работает корректно!")


Тест 4.2: Текстовый запрос

[STATE] math -> classify
[TOOL] memory params={"action": "save", "key": "request_type", "value": "text"}
[STATE] classify -> text
[TOOL] think params={"thought": "Need to greet and ask how user is."}

💭 Размышление агента: Need to greet and ask how user is.
[TOOL] memory params={"action": "save", "key": "response", "value": "Привет! Я в порядке, спасибо, а как у тебя?"}

Результат:
Привет! Я в порядке, спасибо, а как у тебя?

ОБРАБОТКА_ЗАВЕРШЕНА

✓ Текстовый путь работает

✅ Router Agent работает корректно!


---

## 5️⃣ Multi-Agent System - Композиция агентов

Граф: `Supervisor [delegate] → [aggregate] → END`
         ↓ вызывает ↓
       Test Agent, Router Agent

Проверяем:
- Регистрацию агентов в реестре
- Вызов агента из агента через инструмент call_agent
- Агрегацию результатов от нескольких агентов

In [15]:
print("=" * 60)
print("ТЕСТ 5: Multi-Agent System (композиция агентов)")
print("=" * 60)

# Подготовка
reset_memory()
tools_dict = get_tools_dict(tools)

# Создаем подчиненных агентов
test_agent_sub = TestAgent(llm, tools_dict, agent_id="test_agent_sub")
router_agent_sub = RouterAgent(llm, tools_dict, agent_id="router_agent_sub")

print("\n✓ Подчиненные агенты созданы:")
print(f"  - {test_agent_sub}")
print(f"  - {router_agent_sub}")

# Регистрируем их в реестре
register_agent("test_agent", test_agent_sub)
register_agent("router_agent", router_agent_sub)

print("\n✓ Агенты зарегистрированы в реестре")

ТЕСТ 5: Multi-Agent System (композиция агентов)

✓ Подчиненные агенты созданы:
  - TestAgent(id=test_agent_sub, states=1)
  - RouterAgent(id=router_agent_sub, states=4)
[REGISTRY] Зарегистрирован агент: test_agent
[REGISTRY] Зарегистрирован агент: router_agent

✓ Агенты зарегистрированы в реестре


In [16]:
# Создаем супервизора с инструментом call_agent
supervisor_tools_dict = get_tools_dict(tools + multiagent_tools)
supervisor = SupervisorAgent(llm, supervisor_tools_dict)

print(f"✓ Супервизор создан: {supervisor}")

# Визуализация графа
print("\nСтруктура графа супервизора:")
print(supervisor.visualize())

✓ Супервизор создан: SupervisorAgent(id=SupervisorAgent_2150757407040, states=2)

Структура графа супервизора:
📊 Граф агента:

Состояния:
  • delegate (entry)
    Инструменты: call_agent, memory, think
    Описание: Делегирование задач специализированным агентам
  • aggregate
    Инструменты: memory, summarize, think
    Описание: Агрегация результатов от агентов

Переходы:
  • delegate → aggregate (условный)
    Переход к агрегации после делегирования всех задач
  • aggregate → END (условный)
    Завершение после агрегации результатов


In [17]:
# Запуск супервизора
print("\n" + "=" * 60)
print("Запуск: Посчитай 10 * 5 и поздоровайся")
print("=" * 60 + "\n")

result = supervisor.invoke([
    "Посчитай 10 * 5 через test_agent и передай приветствие через router_agent"
])

# Вывод результата
print("\n" + "=" * 60)
print("Результат:")
print("=" * 60)
print(result['messages'][-1].content)

print("\n✅ Multi-Agent System работает корректно!")


Запуск: Посчитай 10 * 5 и поздоровайся

[STATE] START -> delegate
[TOOL] think params={"thought": "Need to call test_agent for calculation and router_agent for greeting."}

💭 Размышление агента: Need to call test_agent for calculation and router_agent for greeting.
[TOOL] call_agent params={"agent_name": "test_agent", "query": "10 * 5"}
[STATE] START -> work
[TOOL] calculator params={"expression": "10 * 5"}
[TOOL] memory_append params={"text": "[calculator] 10 * 5 = 50"}
[TOOL] call_agent params={"agent_name": "router_agent", "query": "Привет!"}
[STATE] START -> classify
[TOOL] think params={"thought": "User says 'Привет!' which is greeting, not math."}

💭 Размышление агента: User says 'Привет!' which is greeting, not math.
[TOOL] memory params={"action": "save", "key": "request_type", "value": "text"}
[STATE] classify -> text
[TOOL] think params={"thought": "We need a friendly greeting and note that we will store the response."}

💭 Размышление агента: We need a friendly greeting and 

---

## 6️⃣ Audit Agent - Полный тест

Граф: `[start_work] → [analize_word] → [analize_sql] → [analize_py] → [self_check] ⟳ → [write_report] → END`

Проверяем:
- Сложный многошаговый workflow
- Роутер с самопроверкой
- Работу с файловой системой
- Анализ разных типов файлов

In [6]:
print("=" * 60)
print("ТЕСТ 6: Audit Agent (полный тест)")
print("=" * 60)

# Подготовка
reset_memory()
tools_dict = get_tools_dict(tools)

# Создание агента
audit_agent = AuditAgent(llm, tools_dict)
print(f"\n✓ Агент создан: {audit_agent}")

# Визуализация графа
print("\nСтруктура графа:")
print(audit_agent.visualize())

ТЕСТ 6: Audit Agent (полный тест)

✓ Агент создан: AuditAgent(id=AuditAgent_3068864894768, states=6)

Структура графа:
📊 Граф агента:

Состояния:
  • start_work (entry)
    Инструменты: ask_human, think, memory, list_data_folders, find_case_folder, list_case_files
    Описание: Поиск папки проверки и сбор списка файлов
  • analize_word
    Инструменты: read_docx_structure, memory, think
    Описание: Анализ docx файлов: реплики и таблицы
  • analize_sql
    Инструменты: read_sql_file, memory, think
    Описание: Анализ SQL скриптов на риски и проблемы
  • analize_py
    Инструменты: read_py_file, memory, think
    Описание: Анализ Python скриптов на риски и проблемы
  • self_check
    Инструменты: memory, think
    Описание: Проверка полноты обработки и выбор следующего шага
  • write_report
    Инструменты: memory, summarize
    Описание: Формирование итогового отчета

Переходы:
  • start_work → analize_word (условный)
    Переход после успешного выбора папки и сбора файлов
  • analiz

In [7]:
# Запуск агента
print("\n" + "=" * 60)
print("Запуск: Аудит папки проверки")
print("=" * 60 + "\n")

print("\nВнимание: Агент будет задавать вопросы!")
print("Попробуйте ввести: 99-41116 или 99-41158\n")

result = audit_agent.invoke([
    "Проведи аудит файлов проверки"
])

# Вывод результата
print("\n" + "=" * 60)
print("Результат:")
print("=" * 60)
print(result['messages'][-1].content)

print("\n✅ Audit Agent работает корректно!")


Запуск: Аудит папки проверки


Внимание: Агент будет задавать вопросы!
Попробуйте ввести: 99-41116 или 99-41158

[STATE] START -> start_work
[TOOL] ask_human params={"question": "Какой номер проверки вы хотите проанализировать?"}

🤔 Вопрос агента: Какой номер проверки вы хотите проанализировать?
[TOOL] list_data_folders params={"data_root": "data"}
[TOOL] find_case_folder params={"case_input": "99-41116", "data_root": "data"}
[TOOL] list_case_files params={"case_folder": "data\\99-41116"}
[TOOL] memory params={"action": "save", "key": "case_id", "value": "99-41116"}
[TOOL] memory params={"action": "save", "key": "case_folder", "value": "data\\99-41116"}
[TOOL] memory params={"action": "save", "key": "case_files", "value": "analysis_01.py,analysis_02.py,audit_report_99-41116.docx,audit_report_summary.txt,query_01.sql,query_02.sql"}
[TOOL] memory params={"action": "save", "key": "docx_files", "value": "audit_report_99-41116.docx"}
[TOOL] memory params={"action": "save", "key": "sql_file

GraphRecursionError: Recursion limit of 25 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/GRAPH_RECURSION_LIMIT

---

## 🎉 Итоги тестирования

Если все тесты пройдены, значит фреймворк агентов работает корректно!

### Что было протестировано:

✅ **Базовые функции** - инструменты работают  
✅ **Test Agent** - простейший агент с 1 состоянием  
✅ **My Agent** - переходы между состояниями  
✅ **Router Agent** - условный роутинг  
✅ **Multi-Agent** - композиция агентов  
✅ **Audit Agent** - сложный workflow  

### Архитектура:

- **AgentConfig** - базовый класс для всех агентов
- **State** - декларативное описание состояний
- **Transition** - описание переходов
- **Изолированная история** - каждый агент имеет свою историю
- **Глобальная память** - инструменты memory доступны всем
- **Реестр агентов** - для мультиагентных систем

### Следующие шаги:

1. Создавайте новых агентов копированием папки и редактированием `agent.py`
2. Используйте композицию через `register_agent()` и `call_agent()`
3. Настраивайте логирование через `enable_logging()` / `disable_logging()`

In [ ]:
print("=" * 60)
print("🎉 ВСЕ ТЕСТЫ ПРОЙДЕНЫ УСПЕШНО!")
print("=" * 60)
print("\nФреймворк агентов готов к использованию!")
print("\nДокументация: README.md")
print("Примеры агентов: agents/*/agent.py")

🎉 ВСЕ ТЕСТЫ ПРОЙДЕНЫ УСПЕШНО!

Фреймворк агентов готов к использованию!

Документация: README.md
Примеры агентов: agents/*/agent.py
